# Build a multi-agent system: the claim desk

*Applied AI for Business · University of Amsterdam*

In this notebook you run a small team of AI agents. Together they handle a hail-damage claim for a fictional insurer. **You do not need to know how to program.** Every step is explained.

**The team**

```
TRIGGER   a claim email arrives
AGENT 1   intake: reads the email and lists the facts
CHECK     plain code: are the facts complete? If not, the chain stops
AGENT 2   policy: looks up the policy with a TOOL and decides
AGENT 3   letter: writes the reply to the customer
PERSON    you: approve the letter or not. Nothing is sent automatically
```

**How to use this notebook**

- A notebook has two kinds of blocks: **text blocks** like this one, and **code blocks** (the grey ones).
- To run a code block: click on it, then click the ▶ button on its left, or press **Shift + Enter**.
- Run the blocks **in order, from top to bottom**. If something goes wrong, choose *Runtime → Restart session* and start again at the top.
- In a code block, text after a `#` is a **comment**: a note for people. The computer ignores it.
- Some blocks show a form on the right (a tick box or a menu). You can use the form instead of changing the code.

**Costs**

- **Demo mode** (the default): free, no account needed. The agents give fixed example answers.
- **Real mode**: uses a real AI model. You need your own OpenAI API key. One run costs well under 1 cent.

All people, policies and claims in this notebook are invented.

## Step 0 · Choose demo mode or real mode

**What this block does:** it stores one choice: demo mode or real mode.

**What you will see:** one line that confirms your choice.

**Cost:** free.

**Want real mode?**
1. Get an API key from your own OpenAI account. Set a monthly spending limit in that account first.
2. In Colab, click the **key icon** in the left bar. Choose *Add new secret*. Name: `OPENAI_API_KEY`. Value: your key. Switch on *Notebook access*.
3. Untick `DEMO_MODE` in the form on the right, and run the block.

⚠️ Never paste your key into a code block. Anyone who sees the notebook would see your key. A Colab secret stays hidden.

In [ ]:
# @title Step 0 · Demo mode or real mode?
DEMO_MODE = True  # @param {type:"boolean"}

# A "variable" is a name that holds a value.
# DEMO_MODE holds True (demo mode) or False (real mode).
if DEMO_MODE:
    print("Demo mode: the agents give fixed example answers. Free.")
else:
    print("Real mode: the agents use a real OpenAI model. You need the secret OPENAI_API_KEY.")

## Step 1 · Connect to the model, and set the budget

**What this block does:**
- It sets the **budget** for one run: how many calls to the model are allowed, and how much a run may cost.
- In real mode, it installs the OpenAI package (software that lets Python send messages to OpenAI's models) and connects with your key from the secret.

**What you will see:** in demo mode, *No connection needed*. In real mode, *Connected to OpenAI*.

**Cost:** free.

In [ ]:
# @title Step 1 · Connect and set the budget

# --- Which model, and what it costs (OpenAI's price list, November 2026) ---
MODEL = "gpt-6-luna"             # a small, cheap model
PRICE_INPUT_PER_MILLION = 0.10   # dollars per 1 million tokens the model READS (a token is about 3/4 of a word)
PRICE_OUTPUT_PER_MILLION = 0.50  # dollars per 1 million tokens the model WRITES (including its thinking)

# --- The budget for one run: the program stops if it would go over ---
MAX_CALLS = 6                    # a normal run needs 4 calls to the model
MAX_COST_USD = 0.05              # a normal run costs well under 1 cent

if DEMO_MODE:
    client = None                # in demo mode we do not connect to anything
    print("No connection needed in demo mode.")
else:
    !pip install -q "openai>=3.51,<4"
    # The line above installs the OpenAI package. The "!" means: run this as a command, not as Python.
    from google.colab import userdata         # Colab's safe place for secrets
    from openai import OpenAI                 # the tool that talks to OpenAI
    try:
        key = userdata.get("OPENAI_API_KEY")  # read your key from the secret, never from the code
    except Exception:
        raise SystemExit("No secret OPENAI_API_KEY found, or 'Notebook access' is off. See Step 0.")
    client = OpenAI(api_key=key)              # the connection, used by every agent below
    print("Connected to OpenAI. Model:", MODEL)

## Step 2 · The data: claim emails and a policy database

**What this block does:** it stores three invented claim emails and a small invented database with two insurance policies.
- Claim 1: hail damage to a roof, with a home insurance policy.
- Claim 2: hail damage to a roof, but the customer only has contents insurance.
- Claim 3: a vague email without a policy number.

**What you will see:** a short list of what was stored.

**Cost:** free.

**Try later:** change the text of an email, run this block again, and run the team again (real mode).

In [ ]:
# @title Step 2 · The claim emails and the policy database

# The three claim emails. Each email has a name (for example "claim_1_covered") and a text.
CLAIMS = {
    "claim_1_covered": """
From: Sanne de Vries
Subject: Hail damage to my roof

Dear claims desk,

During the hailstorm on 3 November 2026, hailstones broke six roof tiles and cracked the skylight above our stairs.
Rain is now coming in. A roofer came by yesterday. His quote for new tiles and a new skylight is EUR 1,850.

My policy number is DH-1001.

Kind regards,
Sanne de Vries
""",
    "claim_2_not_covered": """
From: Mehmet Yilmaz
Subject: Roof and skylight damaged by hail

Hello,

On 3 November 2026 a heavy hailstorm hit our street. The hail damaged our roof and broke the skylight in the attic.
The repair will cost about EUR 2,400 according to the builder.

Policy number: DH-2002.

Thank you,
Mehmet Yilmaz
""",
    "claim_3_missing_info": """
From: Lisa
Subject: storm

hi, after the storm last week there is water coming through the ceiling.
can you help me? what do i need to do?

Lisa
""",
}

# The policy database: two invented policies. Each policy has a number (for example "DH-1001").
POLICIES = {
    "DH-1001": {
        "holder": "Sanne de Vries",
        "product": "Home insurance: building and contents",
        "covers": [
            "storm",
            "hail",
            "fire",
            "water damage from a burst pipe"
        ],
        "does_not_cover": [
            "wear and tear",
            "damage caused by poor maintenance"
        ],
        "deductible_eur": 250,
        "max_payout_eur": 25000,
        "valid_until": "2027-12-31"
    },
    "DH-2002": {
        "holder": "Mehmet Yilmaz",
        "product": "Contents insurance only (the building itself is not insured)",
        "covers": [
            "theft",
            "fire",
            "storm or hail damage to belongings inside the home"
        ],
        "does_not_cover": [
            "damage to the building itself, such as the roof, windows or skylights"
        ],
        "deductible_eur": 100,
        "max_payout_eur": 10000,
        "valid_until": "2027-06-30"
    }
}

print("Stored", len(CLAIMS), "claim emails:", ", ".join(CLAIMS))
print("Stored", len(POLICIES), "policies:", ", ".join(POLICIES))

## Step 3 · The agents' instructions

An agent is mostly its **instructions**: a text that tells the model who it is and what it must do. Below are the instructions of the three agents. The model is the same for all three. Only the instructions differ.

**What these blocks do:** each one stores the instructions of one agent under a name, for example `INTAKE_INSTRUCTIONS`.

**What you will see:** one short confirmation per block.

**Cost:** free.

**Try later:** change the words, run the block again, then run the team again (real mode only: in demo mode the answers are fixed).

In [ ]:
# @title Agent 1 · intake: its instructions

# Everything between the triple quotes is the text the model receives as its instructions.
INTAKE_INSTRUCTIONS = """
You are the INTAKE AGENT of the claims desk of a home insurer.

You read one claim email from a customer and list the facts.

Answer only with JSON, with exactly these keys:
- "customer_name"
- "policy_number"
- "date_of_damage"
- "what_happened"
- "damage"
- "estimated_cost_eur" (a number)

Use null when the email does not say something. Do not guess. Do not add any other text.
"""

print("Stored the instructions of the intake agent.")

In [ ]:
# @title Agent 2 · policy: its instructions

# Everything between the triple quotes is the text the model receives as its instructions.
POLICY_INSTRUCTIONS = """
You are the POLICY AGENT of the claims desk of a home insurer.

You get the facts of a claim as JSON.

1. Use the tool lookup_policy to find the customer's policy.
2. Decide: "COVERED", "NOT COVERED" or "NEED MORE INFORMATION".
   Base your decision only on the policy that the tool returns.
3. If covered, the payout is the estimated cost minus the deductible, and never more than the maximum payout.

Answer only with JSON, with exactly these keys:
- "decision"
- "reason" (one or two short sentences, in plain English)
- "payout_eur" (a number; 0 if not covered; null if unknown)

Do not add any other text.
"""

print("Stored the instructions of the policy agent.")

In [ ]:
# @title Agent 3 · letter: its instructions

# Everything between the triple quotes is the text the model receives as its instructions.
LETTER_INSTRUCTIONS = """
You are the LETTER AGENT of the claims desk of a home insurer.

You get the facts of a claim and the decision of the policy agent, as JSON.

Write a short, friendly reply to the customer, in English:
- Start with the line: DRAFT - to be checked by a claims handler
- Address the customer by name.
- Explain the decision and the reason in plain words.
- Mention the payout only if the decision is COVERED.
- Promise nothing that is not in the decision.
- At most 120 words.
- Sign as: Claims desk, Demo Home Insurance (a fictional insurer)
"""

print("Stored the instructions of the letter agent.")

## Step 4 · The tool

A **tool** is a normal piece of code that does one job. Here it looks up a policy in the database.

The model **cannot run code itself**. It can only *ask* our program to run the tool. Our program runs it and sends the answer back to the model. That is how an agent acts.

**What this block does:**
- It writes down the tool (a *function*: a recipe with a name, here `lookup_policy`).
- It writes a short description of the tool for the model, so the model knows the tool exists and what to send to it.
- It tests the tool once.

**What you will see:** the policy DH-1001, found by the tool.

**Cost:** free.

In [ ]:
# @title Step 4 · The tool: look up a policy
import json   # a standard tool for reading and writing data in JSON format

# The tool itself: a recipe (a "function"). Lines starting with "def" write down a recipe.
def lookup_policy(policy_number):
    """Look up a policy in the database. If the number is unknown, say so."""
    return POLICIES.get(policy_number, {"error": "No policy found with number " + str(policy_number)})

# The description the model receives: the tool's name, what it does, and what to send to it.
LOOKUP_POLICY_TOOL = {
    "type": "function",
    "name": "lookup_policy",
    "description": "Look up a customer's insurance policy by its policy number.",
    "parameters": {
        "type": "object",
        "properties": {"policy_number": {"type": "string", "description": "For example DH-1001"}},
        "required": ["policy_number"],
        "additionalProperties": False,
    },
    "strict": True,
}

# A test: use the recipe once and show the result.
print("Test of the tool, policy DH-1001:")
print(json.dumps(lookup_policy("DH-1001"), indent=2))

## Step 5 · The engine room

This block holds the machinery that every agent uses:
- **the budget guard**: it counts the calls and the cost, and stops a run that would go over the budget;
- **`ask_model`**: sends one message to the model and returns the answer;
- **`read_json`**: reads the model's answer as data;
- **the fixed answers for demo mode.**

You do not need to read this block. The code is hidden; click *Show code* if you are curious.

**What you will see:** *Engine room ready.*

**Cost:** free.

In [ ]:
# @title Step 5 · The engine room (click "Show code" to see the code)

calls = []   # the budget guard's notebook: one line per call to the model

def reset_budget():
    """Start a new run: forget the calls of the previous run."""
    calls.clear()

def total_cost():
    return sum(cost for agent, read, written, cost in calls)

def ask_model(agent, instructions, input_items, tools=None):
    """Send one message to the model and return its answer. The budget guard checks before and after."""
    if len(calls) >= MAX_CALLS:
        raise RuntimeError(f"Budget guard: already {len(calls)} calls in this run (limit {MAX_CALLS}). "
                           "Start a new run with the trigger block.")
    answer = client.responses.create(
        model=MODEL,
        instructions=instructions,          # the agent's instructions (Step 3)
        input=input_items,                  # the work: an email, facts, or a decision
        tools=tools or [],                  # the tools the agent may ask for (only agent 2 has one)
        max_output_tokens=2000,             # the most one answer may use
        reasoning={"effort": "low"},        # less thinking: fewer tokens, so cheaper
    )
    read, written = answer.usage.input_tokens, answer.usage.output_tokens
    cost = (read * PRICE_INPUT_PER_MILLION + written * PRICE_OUTPUT_PER_MILLION) / 1_000_000
    calls.append((agent, read, written, cost))
    if total_cost() > MAX_COST_USD:
        raise RuntimeError(f"Budget guard: this run cost ${total_cost():.4f}, over the limit of ${MAX_COST_USD}.")
    return answer

def read_json(text):
    """The agents answer in JSON. This turns the text into data, or stops with a clear message."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("The model did not answer in JSON. Its answer was: " + text[:300])
    return json.loads(text[start:end + 1])

def show(data):
    """Print data neatly, one fact per line."""
    print(json.dumps(data, indent=2, ensure_ascii=False))

# The fixed answers for demo mode. In real mode, the model writes these answers itself.
DEMO_ANSWERS = {
    "claim_1_covered": {
        "intake": "{\"customer_name\": \"Sanne de Vries\", \"policy_number\": \"DH-1001\", \"date_of_damage\": \"2026-11-03\", \"what_happened\": \"A hailstorm broke roof tiles and cracked a skylight; rain is coming in.\", \"damage\": \"Six broken roof tiles and a cracked skylight\", \"estimated_cost_eur\": 1850}",
        "policy": "{\"decision\": \"COVERED\", \"reason\": \"Hail damage to the building is covered by policy DH-1001. The deductible of EUR 250 applies.\", \"payout_eur\": 1600}",
        "letter": "DRAFT - to be checked by a claims handler\n\nDear Sanne de Vries,\n\nThank you for reporting the hail damage of 3 November 2026. We are sorry to hear that rain is coming in.\n\nYour home insurance covers hail damage to the building. Your roofer's quote is EUR 1,850. After your deductible of EUR 250, we will pay EUR 1,600.\n\nYou can ask your roofer to start the repair.\n\nKind regards,\nClaims desk, Demo Home Insurance (a fictional insurer)"
    },
    "claim_2_not_covered": {
        "intake": "{\"customer_name\": \"Mehmet Yilmaz\", \"policy_number\": \"DH-2002\", \"date_of_damage\": \"2026-11-03\", \"what_happened\": \"Hail damaged the roof and broke the attic skylight.\", \"damage\": \"Damaged roof and broken skylight\", \"estimated_cost_eur\": 2400}",
        "policy": "{\"decision\": \"NOT COVERED\", \"reason\": \"Policy DH-2002 insures only belongings inside the home. Damage to the roof and skylight is damage to the building, which is not insured.\", \"payout_eur\": 0}",
        "letter": "DRAFT - to be checked by a claims handler\n\nDear Mehmet Yilmaz,\n\nThank you for reporting the hail damage of 3 November 2026.\n\nYour policy is a contents insurance. It covers your belongings inside the home, but not the building itself. The roof and the skylight are part of the building, so we cannot pay for this repair.\n\nIf your belongings were damaged as well, please let us know.\n\nKind regards,\nClaims desk, Demo Home Insurance (a fictional insurer)"
    },
    "claim_3_missing_info": {
        "intake": "{\"customer_name\": \"Lisa\", \"policy_number\": null, \"date_of_damage\": null, \"what_happened\": \"After a storm, water is coming through the ceiling.\", \"damage\": \"Water coming through the ceiling\", \"estimated_cost_eur\": null}",
        "policy": "",
        "letter": ""
    }
}

print("Engine room ready.")

## Step 6 · Run the team, one step at a time

Now the team goes to work. Each block below is one step. Run them in order.

Each agent block does two things:
1. it writes down a **recipe** for the agent (the lines that start with `def`);
2. it **uses** the recipe once, for the claim you chose (the last lines of the block).

### The trigger: a claim email arrives

**What this block does:** you choose one of the three claims in the menu on the right. The block starts a new run and shows the email.

**What you will see:** the email.

**Cost:** free.

In [ ]:
# @title The trigger: choose a claim email
CLAIM = "claim_1_covered"  # @param ["claim_1_covered", "claim_2_not_covered", "claim_3_missing_info"]

reset_budget()                 # a new run: the budget guard starts counting from zero
print("A claim email arrives:", CLAIM)
print("-" * 60)
print(CLAIMS[CLAIM])

### Agent 1 · intake: reads the email and lists the facts

**What happens:** the program sends two things to the model: the intake instructions and the email. The model answers with the facts, in a fixed format called JSON: a list of labels with their values.

**What you will see:** the facts, one per line. `null` means: the email does not say.

**Cost in real mode:** 1 call to the model.

In [ ]:
# @title Agent 1 · intake

def intake_agent(claim):
    """The recipe for agent 1: send its instructions and the email to the model, and read the answer as facts."""
    if DEMO_MODE:
        return read_json(DEMO_ANSWERS[claim]["intake"])                 # demo: a fixed example answer
    answer = ask_model("intake agent",                                  # who is asking (for the cost list)
                       INTAKE_INSTRUCTIONS,                             # the agent's instructions
                       [{"role": "user", "content": CLAIMS[claim]}])    # the work: the email
    return read_json(answer.output_text)                                # the model's answer, read as facts

# Use the recipe for the claim you chose:
facts = intake_agent(CLAIM)
print("The facts, according to the intake agent:")
show(facts)

### The check: are the facts complete?

**What happens:** plain code, no AI, checks that the important facts are there. If something is missing, the chain stops here and a person takes over.

This is the lesson from the lecture: mistakes multiply over the steps, and **a check between the steps breaks the chain**.

**What you will see:** *OK*, or a list of what is missing.

**Cost:** free (no AI).

In [ ]:
# @title The check (plain code, no AI)

def check_facts(facts):
    """The recipe for the check: make a list of the important facts that are missing."""
    problems = []
    for label in ["customer_name", "policy_number", "date_of_damage", "estimated_cost_eur"]:
        if not facts.get(label):                  # empty or null counts as missing
            problems.append(label)
    return problems

problems = check_facts(facts)
if problems:
    print("STOP. Missing:", ", ".join(problems))
    print("The chain stops here. A person contacts the customer. Agents 2 and 3 will not run.")
else:
    print("OK. All important facts are there. The work goes to agent 2.")

### Agent 2 · policy: uses the tool, then decides

**What happens, step by step:**
1. The program sends the model the policy instructions, the facts, and the description of the tool.
2. The model answers with a **request**: *please run lookup_policy for policy DH-1001*.
3. Our program runs the tool and sends the result back.
4. The model reads the policy and gives its decision.

**What you will see:** the tool request, the tool's answer, and the decision.

**Cost in real mode:** 2 calls to the model (one before the tool, one after).

In [ ]:
# @title Agent 2 · policy (with the tool)

def policy_agent(facts):
    """The recipe for agent 2: let the model use the tool, then read its decision."""
    if DEMO_MODE:
        print("The policy agent asks for the tool: lookup_policy('" + facts["policy_number"] + "')")
        print("The tool answers:", lookup_policy(facts["policy_number"])["product"])   # the real tool still runs
        return read_json(DEMO_ANSWERS[CLAIM]["policy"])                              # demo: a fixed decision

    conversation = [{"role": "user", "content": json.dumps(facts)}]   # what the agent gets first: the facts
    for attempt in range(3):                                          # at most 3 rounds, then give up
        answer = ask_model("policy agent", POLICY_INSTRUCTIONS, conversation, tools=[LOOKUP_POLICY_TOOL])
        conversation += answer.output                                 # remember what the model said
        requests = [item for item in answer.output if item.type == "function_call"]
        if not requests:                                              # no tool request: so this is the decision
            return read_json(answer.output_text)
        for request in requests:                                      # the model ASKS ...
            policy_number = json.loads(request.arguments)["policy_number"]
            print("The policy agent asks for the tool: lookup_policy('" + policy_number + "')")
            result = lookup_policy(policy_number)                     # ... and OUR CODE runs the tool
            print("The tool answers:", result.get("product", result))
            conversation.append({"type": "function_call_output",      # send the tool's answer back
                                 "call_id": request.call_id,
                                 "output": json.dumps(result)})
    raise RuntimeError("The policy agent did not decide after 3 rounds.")

# Use the recipe, but only if the check said OK:
if problems:
    decision = None
    print("Skipped: the check stopped the chain.")
else:
    decision = policy_agent(facts)
    print()
    print("The decision of the policy agent:")
    show(decision)

### Agent 3 · letter: writes the reply

**What happens:** the program sends the model the letter instructions, the facts and the decision. The model writes a draft reply.

**What you will see:** the draft letter.

**Cost in real mode:** 1 call to the model.

In [ ]:
# @title Agent 3 · letter

def letter_agent(facts, decision):
    """The recipe for agent 3: send its instructions, the facts and the decision; return the draft letter."""
    if DEMO_MODE:
        return DEMO_ANSWERS[CLAIM]["letter"]                            # demo: a fixed example letter
    work = json.dumps({"facts": facts, "decision": decision})           # the work: facts + decision together
    answer = ask_model("letter agent", LETTER_INSTRUCTIONS, [{"role": "user", "content": work}])
    return answer.output_text                                           # the letter, as plain text

# Use the recipe, but only if there is a decision:
if decision is None:
    letter = None
    print("Skipped: there is no decision.")
else:
    letter = letter_agent(facts, decision)
    print(letter)

### The person: you decide

**What happens:** nothing is sent automatically. You are the claims handler. You read the draft and decide.

**What you will see:** the letter, and a box below it. Type `yes` or `no` in the box and press Enter.

**Cost:** free.

In [ ]:
# @title The person: approve or not?

if letter is None:
    print("Nothing to approve: the chain stopped earlier. A person contacts the customer.")
else:
    print(letter)
    print()
    choice = input("Approve this letter? Type yes or no, then press Enter: ").strip().lower()
    if choice == "yes":
        print("Approved. In a real system, the letter would now be sent. Here, nothing is sent.")
    else:
        print("Not approved. The claims handler changes the letter, or asks a colleague.")

### What did this run cost?

**What you will see:** one line per call to the model, and the total. In demo mode there were no calls, so the cost is zero.

In [ ]:
# @title The cost of this run

print(f"{'Agent':<15}{'Tokens read':>13}{'Tokens written':>16}{'Cost (USD)':>13}")
for agent, read, written, cost in calls:
    print(f"{agent:<15}{read:>13}{written:>16}{cost:>13.5f}")
print(f"Total: {len(calls)} calls, ${total_cost():.5f}")

## Step 7 · The glue: the whole team in one go

In Step 6 you ran the agents one by one, so you could see each step. In a real system, a few lines of code run them in a row. That code is the **glue**.

**What this block does:** it runs one claim from start to finish, using the recipes from Step 6. The person step is left out: the draft waits for approval.

**What you will see:** each step, the draft letter (if the chain gets that far), and the cost.

**Cost in real mode:** up to 4 calls, well under 1 cent.

Notice that the order is fixed in the code: intake, check, policy, letter. The model never decides who goes next. That makes this a **workflow**. If a model decided the next step, it would be an **agent** in the strict sense.

In [ ]:
# @title Step 7 · The glue
RUN_CLAIM = "claim_2_not_covered"  # @param ["claim_1_covered", "claim_2_not_covered", "claim_3_missing_info"]

def run_claim_desk(claim):
    """The glue: the trigger, then the agents and the check, in a fixed order."""
    global CLAIM
    CLAIM = claim                                  # the demo answers look up the claim by this name
    reset_budget()
    print("TRIGGER   a claim email arrives:", claim)
    facts = intake_agent(claim)                    # agent 1
    print("AGENT 1   facts:", facts)
    problems = check_facts(facts)                  # the check
    if problems:
        print("CHECK     STOP. Missing:", ", ".join(problems), "-> a person takes over.")
        return
    print("CHECK     OK")
    decision = policy_agent(facts)                 # agent 2 (with the tool)
    print("AGENT 2   decision:", decision["decision"], "-", decision["reason"])
    letter = letter_agent(facts, decision)         # agent 3
    print("AGENT 3   draft letter:")
    print(letter)
    print("PERSON    the draft waits for a claims handler. Nothing has been sent.")
    print(f"COST      {len(calls)} calls, ${total_cost():.5f}")

run_claim_desk(RUN_CLAIM)

## Try this

Work in pairs. Write down your prediction **before** you run anything.

1. **Claim 3 stops at the check.** Why? Add the missing facts to the email in Step 2, run Step 2 again, and run the team again (real mode). Does it get through now?
2. **Change one sentence** in the letter instructions (Step 3), for example: *write the letter in Dutch* or *write in a very formal style*. Run Step 3 and agent 3 again. What changes?
3. **Make the check stricter.** In the check block, stop the chain when `estimated_cost_eur` is above 5,000, so a person must decide on large claims. (Hint: ask ChatGPT or Claude how to add that line, and paste it in.)
4. **Run the same claim five times** in real mode (Step 7). Is the decision the same every time? Is the letter the same? What does that mean for a firm?
5. **Add a fourth agent:** a fraud agent that runs after the check and flags claims that look odd. You need new instructions (Step 3), a recipe like the other agents (Step 6), and one extra line in the glue (Step 7).
6. **Discuss:** which steps would you never let an agent do without a person? Which level of autonomy (L1 to L5, from the lecture) does this claim desk have?

## What you have seen

| Lecture term | Where in this notebook |
|---|---|
| Agent = instructions + model + tools | Step 3 (instructions), Step 1 (model), Step 4 (tool) |
| The model asks, the code acts | Agent 2: the tool request and the tool's answer |
| A check breaks the chain of mistakes | The check block |
| Human in the loop | The person block |
| Workflow: the code decides the order | Step 7, the glue |
| Budget and safety | Step 0 (secret), Step 1 (budget), Step 5 (budget guard) |

*Teaching material for Applied AI for Business, Amsterdam Business School, University of Amsterdam. The insurer, its policies and its customers are fictional. This notebook is not connected to any real insurer.*